In [12]:
# Install xgboost in the current Jupyter kernel
%pip install xgboost

  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-3.2.0-py3-none-win_amd64.whl (101.7 MB)
Note: you may need to restart the kernel to use updated packages.


In [13]:
import pandas as pd
import numpy as np
import zipfile
import warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import xgboost as xgb
 


In [14]:

 
BASE = "Datasets/"
 
# 1A. Crop data (area, production, yield per crop) - BACKBONE
crop = pd.read_csv(BASE + "ICRISAT-Crop data.csv")
 
# 1B. Prices - harvest price per crop (has -1 for missing)
price = pd.read_csv(BASE + "ICRISAT-Price.csv")
price.replace(-1, np.nan, inplace=True)  # -1 = missing in ICRISAT convention
 
# 1C. Fertilisers - N, P, K consumption
fert = pd.read_csv(BASE + "ICRISAT-Fertilisers.csv")
 
# 1D. Rainfall (ICRISAT) - monthly, 1990-2020
rain_icrisat = pd.read_csv(BASE + "ICRISAT-Rainfall.csv")
 
# 1E. Landuse - net/gross cropped area, cropping intensity
landuse = pd.read_csv(BASE + "ICRISAT-land-different.csv")
 
# 1F. Irrigated area per crop
area_irr = pd.read_csv(BASE + "ICRISAT-Area.csv")
 
# 1G. Temperature (monthly, Kurnool station, 1969-2020)
temp = pd.read_csv(BASE + "important temperature data.csv")
temp['date'] = pd.to_datetime(temp['date'])
temp['Year'] = temp['date'].dt.year
 
# 1H. Rainfall_again (daily, 2009-2023) - will aggregate to annual
rain2 = pd.read_csv(BASE + "rainfall again.csv")
rain2['date'] = pd.to_datetime(rain2['date'])
rain2['Year'] = rain2['date'].dt.year
 
# 1I. FAOSTAT Import/Export (India-level, 1961-2024)
fao = pd.read_csv(BASE + "FAOSTAT-IMPORTEXPORT.csv")
 
# 1J. CPI - Crop Production Index (India, 1960-2022)
with zipfile.ZipFile(BASE + "CPI.zip") as z:
    with z.open("API_AG.PRD.CROP.XD_DS2_en_csv_v2_11657.csv") as f:
        cpi_raw = pd.read_csv(f, skiprows=4)
cpi_india = cpi_raw[cpi_raw['Country Name'] == 'India'].copy()
year_cols = [c for c in cpi_india.columns if c.isdigit()]
cpi = cpi_india[year_cols].T.reset_index()
cpi.columns = ['Year', 'CPI']
cpi['Year'] = cpi['Year'].astype(int)
cpi['CPI'] = pd.to_numeric(cpi['CPI'], errors='coerce')
cpi = cpi.dropna()


In [15]:
# --- 2A. TEMPERATURE: pivot min/max, compute annual mean ---
# WHY: Temp affects crop stress, evapotranspiration, pest cycles
temp_min = temp[temp['parameter'] == 'minimum temperature'].groupby('Year')['average_temperature'].mean().reset_index()
temp_max = temp[temp['parameter'] == 'maximum temperature'].groupby('Year')['average_temperature'].mean().reset_index()
temp_min.columns = ['Year', 'avg_min_temp_c']
temp_max.columns = ['Year', 'avg_max_temp_c']
temp_annual = temp_min.merge(temp_max, on='Year')
temp_annual['temp_range'] = temp_annual['avg_max_temp_c'] - temp_annual['avg_min_temp_c']
 
# --- 2B. RAINFALL: use ICRISAT monthly → compute kharif/rabi season rainfall ---
# WHY: Kharif (Jun-Sep) and Rabi (Oct-Feb) rainfall matter differently for different crops
# Kharif rainfall → directly affects Sorghum, Groundnut, Pearl Millet yields
# Rabi rainfall → affects Chickpea, Wheat, Pigeonpea
 
kharif_months = ['JUNE RAINFALL (Millimeters)', 'JULY RAINFALL (Millimeters)',
                 'AUGUST RAINFALL (Millimeters)', 'SEPTEMBER RAINFALL (Millimeters)']
rabi_months = ['OCTOBER RAINFALL (Millimeters)', 'NOVEMBER RAINFALL (Millimeters)',
               'DECEMBER RAINFALL (Millimeters)', 'JANUARY RAINFALL (Millimeters)',
               'FEBRUARY RAINFALL (Millimeters)']
 
rain_fe = rain_icrisat[['Year', 'ANNUAL RAINFALL (Millimeters)']].copy()
rain_fe.columns = ['Year', 'annual_rainfall_mm']
rain_fe['kharif_rainfall_mm'] = rain_icrisat[kharif_months].sum(axis=1).values
rain_fe['rabi_rainfall_mm'] = rain_icrisat[rabi_months].sum(axis=1).values
 
# Rainfall lag (t-1 year): lagged rainfall affects yield with a delay
# WHY: Soil moisture recharge from previous kharif affects rabi crops
rain_fe['kharif_rainfall_lag1'] = rain_fe['kharif_rainfall_mm'].shift(1)
rain_fe['annual_rainfall_lag1'] = rain_fe['annual_rainfall_mm'].shift(1)
 
# --- 2C. FERTILISER: NPK ratio + total intensity ---
# WHY: N/P/K ratio is a soil health signal; total per ha reflects input investment
fert_fe = fert[['Year', 'NITROGEN CONSUMPTION (tons)', 'PHOSPHATE CONSUMPTION (tons)',
                 'POTASH CONSUMPTION (tons)', 'TOTAL CONSUMPTION (tons)',
                 'TOTAL PER HA OF GCA (Kg per ha)']].copy()
fert_fe.columns = ['Year', 'N_consumption_tons', 'P_consumption_tons',
                   'K_consumption_tons', 'total_fert_tons', 'fert_per_ha_gca']
 
# NPK ratio features - imbalanced NPK = nutrient stress
fert_fe['N_P_ratio'] = fert_fe['N_consumption_tons'] / (fert_fe['P_consumption_tons'] + 1)
fert_fe['N_K_ratio'] = fert_fe['N_consumption_tons'] / (fert_fe['K_consumption_tons'] + 1)
 
# Fertiliser trend (3-year rolling mean) - smooths out data entry anomalies
fert_fe['fert_rolling3'] = fert_fe['fert_per_ha_gca'].rolling(3).mean()
 
# --- 2D. LANDUSE: cropping intensity, fallow ratio ---
# WHY: Gross/Net Cropped Area ratio = cropping intensity, fallow area = unused land signal
land_fe = landuse[['Year', 'NET CROPPED AREA (1000 ha)', 'GROSS CROPPED AREA (1000 ha)',
                    'CURRENT FALLOW AREA (1000 ha)', 'CROPING INTENSITY (Percent)']].copy()
land_fe.columns = ['Year', 'net_cropped_area', 'gross_cropped_area', 'fallow_area', 'cropping_intensity']
land_fe['fallow_ratio'] = land_fe['fallow_area'] / (land_fe['net_cropped_area'] + 1)
 
# --- 2E. FAOSTAT: India-level import pressure per crop ---
# WHY: When India imports heavily, domestic price drops (excess supply)
# When exports rise, domestic supply tightens → price rises
fao_pivot = fao.pivot_table(index=['Year', 'Item'], columns='Element', values='Value').reset_index()
fao_pivot.columns.name = None
fao_pivot = fao_pivot.rename(columns={
    'Import quantity': 'import_qty_tons',
    'Import value': 'import_val_1000usd',
    'Export quantity': 'export_qty_tons',
    'Export value': 'export_val_1000usd'
})
 
# Net trade balance: positive = net exporter (supply surplus), negative = net importer
fao_pivot['net_export_qty'] = fao_pivot.get('export_qty_tons', 0).fillna(0) - fao_pivot.get('import_qty_tons', 0).fillna(0)
 
# Map FAOSTAT crop names to ICRISAT crop names
crop_map = {
    'Rice': 'RICE', 'Wheat': 'WHEAT', 'Sorghum': 'SORGHUM',
    'Maize (corn)': 'MAIZE', 'Millet': 'PEARL MILLET',
    'Chick peas, dry': 'CHICKPEA', 'Pigeon peas, dry': 'PIGEONPEA',
    'Groundnuts, shelled': 'GROUNDNUT', 'Cotton seed': 'COTTON',
    'Sugar cane': 'SUGARCANE', 'Linseed': 'LINSEED',
    'Castor oil seeds': 'CASTOR', 'Barley': 'BARLEY',
    'Mustard seed': 'RAPESEED AND MUSTARD', 'Safflower seed': 'SAFFLOWER',
    'Sunflower seed': 'SUNFLOWER', 'Soya beans': 'SOYABEAN',
}
fao_pivot['crop_std'] = fao_pivot['Item'].map(crop_map)
 
# --- 2F. CPI: India crop production index - macro signal ---
# WHY: National crop production trend affects Kurnool price via market integration
cpi_fe = cpi.copy()
cpi_fe['cpi_lag1'] = cpi_fe['CPI'].shift(1)
cpi_fe['cpi_yoy_change'] = cpi_fe['CPI'].pct_change() * 100  # year-over-year % change
 
# --- 2G. CYCLICAL TIME ENCODING ---
# WHY: Year as integer is wrong - model doesn't know 1975 is between 1974 and 1976 cyclically
# Use decade encoding and linear trend
# (Full sine/cosine encoding more relevant for monthly data; for annual, use trend + decade)
 
# ============================================================
# STEP 3: MELT CROP DATA TO LONG FORMAT
# ============================================================
 
# WHY: ICRISAT stores one row per year, one column per crop
# For ML: we want one row per (year, crop) - this is the melting/pivoting technique from notes
# Each row = one observation = (Year, Crop, Area, Production, Yield, IrrigatedArea, Price)
 
crop_cols = {}
for col in crop.columns:
    for crop_name in ['RICE', 'WHEAT', 'SORGHUM', 'PEARL MILLET', 'MAIZE',
                       'CHICKPEA', 'PIGEONPEA', 'GROUNDNUT', 'COTTON', 'SUGARCANE',
                       'CASTOR', 'SUNFLOWER', 'FINGER MILLET']:
        if col.startswith(crop_name) and 'KHARIF' not in col and 'RABI' not in col:
            metric = col.replace(crop_name, '').strip()
            if crop_name not in crop_cols:
                crop_cols[crop_name] = {}
            crop_cols[crop_name][metric] = col
 
long_rows = []
for crop_name, metrics in crop_cols.items():
    if 'AREA (1000 ha)' in metrics and 'PRODUCTION (1000 tons)' in metrics and 'YIELD (Kg per ha)' in metrics:
        tmp = crop[['Year', metrics['AREA (1000 ha)'], metrics['PRODUCTION (1000 tons)'], metrics['YIELD (Kg per ha)']]].copy()
        tmp.columns = ['Year', 'area_1000ha', 'production_1000tons', 'yield_kg_ha']
        tmp['crop'] = crop_name
        long_rows.append(tmp)
 
df_long = pd.concat(long_rows, ignore_index=True)
df_long = df_long[df_long['area_1000ha'] > 0].copy()  # drop crops not grown in Kurnool
 
# Add price per crop
price_long = []
price_col_map = {
    'PADDY': 'RICE', 'SORGHUM': 'SORGHUM', 'PEARL MILLET': 'PEARL MILLET',
    'CHICKPEA': 'CHICKPEA', 'PIGEONPEA': 'PIGEONPEA', 'GROUNDNUT': 'GROUNDNUT',
    'COTTON KAPAS': 'COTTON', 'MAIZE': 'MAIZE',
}
for price_key, crop_name in price_col_map.items():
    pcol = [c for c in price.columns if price_key in c and 'PRICE' in c]
    if pcol:
        tmp = price[['Year', pcol[0]]].copy()
        tmp.columns = ['Year', 'harvest_price_rs_quintal']
        tmp['crop'] = crop_name
        price_long.append(tmp)
 
price_df = pd.concat(price_long, ignore_index=True)
df_long = df_long.merge(price_df, on=['Year', 'crop'], how='left')


In [16]:
 
# All environmental and macro features are year-level (not crop-level)
# So we merge on Year - each crop row gets the same env features for that year
 
df = df_long.copy()
df = df.merge(temp_annual, on='Year', how='left')
df = df.merge(rain_fe, on='Year', how='left')
df = df.merge(fert_fe, on='Year', how='left')
df = df.merge(land_fe, on='Year', how='left')
df = df.merge(cpi_fe, on='Year', how='left')
 
# FAOSTAT: add import pressure per crop (India-level signal)
fao_merge = fao_pivot[['Year', 'crop_std', 'net_export_qty']].dropna(subset=['crop_std'])
df = df.merge(fao_merge, left_on=['Year', 'crop'], right_on=['Year', 'crop_std'], how='left')
df.drop(columns=['crop_std'], inplace=True, errors='ignore')
 
# Crop one-hot encoding (RF/XGBoost need numeric crop labels)
df['crop_code'] = df['crop'].astype('category').cat.codes
 
# Year trend feature (linear)
df['year_trend'] = df['Year'] - df['Year'].min()
 
# Yield lag (previous year's yield for same crop - strong autoregressive signal)
df = df.sort_values(['crop', 'Year'])
df['yield_lag1'] = df.groupby('crop')['yield_kg_ha'].shift(1)
df['yield_lag2'] = df.groupby('crop')['yield_kg_ha'].shift(2)
df['yield_rolling3'] = df.groupby('crop')['yield_kg_ha'].transform(
    lambda x: x.shift(1).rolling(3).mean()
)
 
# Price lag
df['price_lag1'] = df.groupby('crop')['harvest_price_rs_quintal'].shift(1)
 
# Area change YoY (farmer planting decision signal)
df['area_yoy_change'] = df.groupby('crop')['area_1000ha'].pct_change() * 100
 
 

In [18]:
# ============================================================
# STEP 5: PREPARE FOR MODELLING
# ============================================================
 
TARGET = 'yield_kg_ha'
 
FEATURES = [
    # Crop identity
    'crop_code',
    # Supply-side
    'area_1000ha', 'area_yoy_change',
    # Yield history (autoregressive)
    'yield_lag1', 'yield_lag2', 'yield_rolling3',
    # Price
    'price_lag1',
    # Fertiliser
    'N_consumption_tons', 'P_consumption_tons', 'K_consumption_tons',
    'fert_per_ha_gca', 'N_P_ratio', 'fert_rolling3',
    # Weather
    'avg_min_temp_c', 'avg_max_temp_c', 'temp_range',
    'annual_rainfall_mm', 'kharif_rainfall_mm', 'rabi_rainfall_mm',
    'kharif_rainfall_lag1', 'annual_rainfall_lag1',
    # Landuse
    'net_cropped_area', 'cropping_intensity', 'fallow_ratio',
    # Macro
    'CPI', 'cpi_lag1', 'cpi_yoy_change',
    'net_export_qty',
    # Time
    'year_trend',
]
 
df_model = df[FEATURES + [TARGET, 'Year', 'crop']].copy()
df_model = df_model.dropna(subset=[TARGET])
 
# Impute remaining missing values (median imputation)
# WHY: XGBoost can handle NaN but RF cannot; impute for fair comparison
imputer = SimpleImputer(strategy='median')
X = df_model[FEATURES].copy()
y = df_model[TARGET].copy()
 
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=FEATURES)
 
 
# Temporal train/test split (NOT random split - time series rule)
# WHY: Random split leaks future data into training (data leakage)
# Always split time series by time - train on past, test on future
split_year = 2010
train_mask = df_model['Year'] <= split_year
test_mask = df_model['Year'] > split_year
 
X_train, X_test = X_imputed[train_mask], X_imputed[test_mask]
y_train, y_test = y[train_mask], y[test_mask]
 


C:\Users\lenovo\AppData\Local\Temp\ipykernel_2148\4055266451.py:51: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  X_train, X_test = X_imputed[train_mask], X_imputed[test_mask]
C:\Users\lenovo\AppData\Local\Temp\ipykernel_2148\4055266451.py:51: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  X_train, X_test = X_imputed[train_mask], X_imputed[test_mask]


In [19]:
# ============================================================
# STEP 6: RANDOM FOREST
# ============================================================
 
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
 
rf_r2   = r2_score(y_test, rf_pred)
rf_mae  = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
 
print(f"Random Forest Results:")
print(f"  R²   = {rf_r2:.4f}  (how much variance explained; >0.7 is good)")
print(f"  MAE  = {rf_mae:.1f} kg/ha (average error in yield prediction)")
print(f"  RMSE = {rf_rmse:.1f} kg/ha")
 
# Feature importance
rf_importance = pd.DataFrame({
    'feature': FEATURES,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)
 
print(f"\nTop 10 most important features (RF):")
print(rf_importance.head(10).to_string(index=False))

Random Forest Results:
  R²   = 0.7352  (how much variance explained; >0.7 is good)
  MAE  = 670.2 kg/ha (average error in yield prediction)
  RMSE = 1304.2 kg/ha

Top 10 most important features (RF):
             feature  importance
          yield_lag1    0.579110
      yield_rolling3    0.274472
          yield_lag2    0.046747
  annual_rainfall_mm    0.018546
         area_1000ha    0.010936
annual_rainfall_lag1    0.007282
           crop_code    0.006146
      avg_max_temp_c    0.005869
  K_consumption_tons    0.005412
        fallow_ratio    0.004569


In [20]:
# ============================================================
# STEP 7: XGBOOST
# ============================================================
 
xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)
xgb_model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)],
              verbose=False)
xgb_pred = xgb_model.predict(X_test)
 
xgb_r2   = r2_score(y_test, xgb_pred)
xgb_mae  = mean_absolute_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
 
print(f"XGBoost Results:")
print(f"  R²   = {xgb_r2:.4f}")
print(f"  MAE  = {xgb_mae:.1f} kg/ha")
print(f"  RMSE = {xgb_rmse:.1f} kg/ha")
 
xgb_importance = pd.DataFrame({
    'feature': FEATURES,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)
 
print(f"\nTop 10 most important features (XGBoost):")
print(xgb_importance.head(10).to_string(index=False))

XGBoost Results:
  R²   = 0.6686
  MAE  = 762.7 kg/ha
  RMSE = 1458.9 kg/ha

Top 10 most important features (XGBoost):
             feature  importance
          yield_lag1    0.285520
      yield_rolling3    0.251861
          yield_lag2    0.090791
annual_rainfall_lag1    0.040835
  annual_rainfall_mm    0.033416
  K_consumption_tons    0.033279
       fert_rolling3    0.025649
      avg_max_temp_c    0.022857
           crop_code    0.022182
kharif_rainfall_lag1    0.018723


In [21]:
# ============================================================
# STEP 8: VISUALISATIONS
# ============================================================
 
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Kurnool Crop Yield Prediction - Model Viability Analysis', fontsize=14, fontweight='bold')
 
# Plot 1: RF Predicted vs Actual
ax = axes[0, 0]
ax.scatter(y_test, rf_pred, alpha=0.6, color='steelblue', s=30)
lim = [min(y_test.min(), rf_pred.min()), max(y_test.max(), rf_pred.max())]
ax.plot(lim, lim, 'r--', lw=1.5, label='Perfect prediction')
ax.set_xlabel('Actual Yield (kg/ha)')
ax.set_ylabel('Predicted Yield (kg/ha)')
ax.set_title(f'Random Forest\nR²={rf_r2:.3f} | MAE={rf_mae:.0f} kg/ha')
ax.legend()
 
# Plot 2: XGBoost Predicted vs Actual
ax = axes[0, 1]
ax.scatter(y_test, xgb_pred, alpha=0.6, color='darkorange', s=30)
ax.plot(lim, lim, 'r--', lw=1.5, label='Perfect prediction')
ax.set_xlabel('Actual Yield (kg/ha)')
ax.set_ylabel('Predicted Yield (kg/ha)')
ax.set_title(f'XGBoost\nR²={xgb_r2:.3f} | MAE={xgb_mae:.0f} kg/ha')
ax.legend()
 
# Plot 3: Model comparison bar chart
ax = axes[0, 2]
models = ['Random Forest', 'XGBoost']
r2s = [rf_r2, xgb_r2]
colors = ['steelblue', 'darkorange']
bars = ax.bar(models, r2s, color=colors, width=0.4)
ax.set_ylabel('R² Score')
ax.set_title('Model Comparison (R²)')
ax.set_ylim(0, 1)
ax.axhline(y=0.7, color='green', linestyle='--', alpha=0.7, label='Good threshold (0.7)')
ax.legend()
for bar, val in zip(bars, r2s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
 
# Plot 4: RF Feature Importance (Top 12)
ax = axes[1, 0]
top12 = rf_importance.head(12)
colors_fi = ['#d62728' if i < 3 else '#1f77b4' for i in range(12)]
ax.barh(range(len(top12)), top12['importance'].values, color=colors_fi)
ax.set_yticks(range(len(top12)))
ax.set_yticklabels(top12['feature'].values, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Importance Score')
ax.set_title('RF Feature Importance (Top 12)\nRed = Top 3 drivers')
 
# Plot 5: XGBoost Feature Importance (Top 12)
ax = axes[1, 1]
top12x = xgb_importance.head(12)
ax.barh(range(len(top12x)), top12x['importance'].values, color='darkorange', alpha=0.8)
ax.set_yticks(range(len(top12x)))
ax.set_yticklabels(top12x['feature'].values, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Importance Score')
ax.set_title('XGBoost Feature Importance (Top 12)')
 
# Plot 6: Yield trend for top 3 crops in Kurnool
ax = axes[1, 2]
top_crops = df_long.groupby('crop')['production_1000tons'].sum().nlargest(4).index
for crop_name in top_crops:
    tmp = df_long[df_long['crop'] == crop_name].sort_values('Year')
    ax.plot(tmp['Year'], tmp['yield_kg_ha'], marker='o', markersize=3,
            label=crop_name, alpha=0.8)
ax.set_xlabel('Year')
ax.set_ylabel('Yield (kg/ha)')
ax.set_title('Yield Trends - Top Crops in Kurnool')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)
 
plt.tight_layout()
plt.savefig('kurnool_viability_analysis.png', dpi=150, bbox_inches='tight')
